# Lab 2 - Övervakadinlärning

I denna laboration ska du utforska **övervakadinlärning** för klassificeringsproblem. Vi börjar med att klassificera irisblommor. Sedan tittar vi på ett mer praktiskt problem: att identifiera bedrägliga kreditkortstransaktioner.

**Gör laboratorierna uppifrån och ned.**

**Du behöver inte läsa vidare än den sektion du arbetar med.**

---

## Vad är övervakadinlärning?

Övervakadinlärning är när vi tränar en maskinlärningsmodell med data som redan har rätt svar.

Exempel:
- Vi ger modellen foton av blommor och berättar vilken art de tillhör
- Modellen lär sig mönster och kan sedan gissa arten på nya foton

I denna lab: **Kan modellen förutsäga rätt svar på data den aldrig sett förut?**

# Del 2.1 - Klassificering av irisblommor

Vi börjar med ett enkelt exempel: Irisdatasettet.

## Vad är Irisdatasettet?

Det är data om 150 irisblommor med 4 mätningar för varje blomma:
- Sepalens längd (yttre kronblad)
- Sepalens bredd
- Kronbladets längd
- Kronbladets bredd

Uppgiften: **Förutsäga vilken art blomman är baserat på dessa mätningar.**

Det finns 3 arter: Setosa, Versicolor och Virginica.

## Steg 1: Importera bibliotek

Vi behöver några verktyg för att arbeta med data och träna modeller.

In [ ]:
# Importera bibliotek
import xgboost as xgb                          # En kraftfull maskinlärningsmodell
import matplotlib.pyplot as plt                # För att rita grafer
import seaborn as sns                          # För snyggare grafer
import pandas as pd                            # För att arbeta med tabeller
from sklearn.datasets import load_iris         # Irisdatasettet
from sklearn.model_selection import train_test_split  # Dela data
from sklearn.metrics import confusion_matrix   # Förvirringmatris

## Steg 2: Ladda och titta på data

Låt oss ladda Irisdatasettet och se hur det ser ut.

In [ ]:
# Ladda Irisdatasettet
iris = load_iris()

# Skapa en tabell (DataFrame) med all data
data = pd.DataFrame(iris.data, columns=iris.feature_names)
data['Art'] = iris.target  # Lägg till arten (0, 1 eller 2)

# Visa de första 10 raderna
print("Första 10 rader av data:")
print(data.head(10))
print(f"\nTotalt antal blommor: {len(data)}")

## Steg 3: Dela data i träning och test

Vi delar data i två delar:
- **Träningsdata** (80%): Modellen lär sig från detta
- **Testdata** (20%): Vi testar hur bra modellen är på ny data

Det är viktigt att modellen **INTE** ser testdatan under träningen!

In [ ]:
# Separera features (mätningar) från target (arten)
X = data.drop('Art', axis=1)  # Features (mätningar)
y = data['Art']               # Target (arten vi vill förutsäga)

# Dela data: 80% träning, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Träningsdata: {len(X_train)} blommor")
print(f"Testdata:     {len(X_test)} blommor")

## Steg 4: Träna modellen

Vi använder XGBoost, en kraftfull maskinlärningsmodell.

**Parametrar förklarade:**
- `max_depth`: Hur djup modellen kan vara (större = mer komplicerad)
- `eta`: Hur snabbt modellen lär sig (inlärningshastighet)
- `num_class`: Antalet klasser (blomarter) = 3
- `num_round`: Hur många gånger modellen uppdateras under träningen

In [ ]:
# Konvertera till XGBoost-format
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# Sätt modellparametrar
param = {
    'max_depth': 4,                    # Modellens djup
    'eta': 0.3,                        # Inlärningshastighet
    'num_class': 3,                    # 3 blomarter
    'objective': 'multi:softprob'      # För klassificering med flera klasser
}

# Träna modellen
model = xgb.train(param, dtrain, num_boost_round=10)

print("Modellen är tränad!")

## Steg 5: Gör förutsägelser

Låt oss se hur bra modellen är på testdata.

In [ ]:
# Förutsäg på testdata
predictions = model.predict(dtest)

# Välj den klass med högst sannolikhet (0, 1 eller 2)
predicted_class = predictions.argmax(axis=1)

# Räkna rätt svar
correct  = (predicted_class == y_test.values).sum()
total    = len(y_test)
accuracy = correct / total

print(f"Rätta svar:  {correct}/{total}")
print(f"Noggrannhet: {accuracy*100:.1f}%")

## Steg 6: Visualisera resultaten med en förvirringmatris

En **förvirringmatris** visar vilka blommor modellen klassificerade rätt och vilka den missade.

- **Diagonalen** (övre vänster till nedre höger) visar rätta svar
- **Övriga rutor** visar fel – t.ex. att modellen trodde det var art 1 fast det var art 2

In [ ]:
# Skapa förvirringmatris
cm = confusion_matrix(y_test, predicted_class)

# Rita den som en värmekarta
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Förvirringmatris – Irisklassificering')
plt.ylabel('Faktisk art')
plt.xlabel('Förutsagd art')
plt.tight_layout()
plt.show()

print("\nMatrisen visar:")
print("  Rad    = den faktiska arten")
print("  Kolumn = den art modellen gissade")
print("  Siffran = antal blommor")

---

# Del 2.2 - Bedrägeridetektering i kreditkort

Nu tittar vi på ett mer komplicerat och verkligt problem: att upptäcka bedrägliga kreditkortstransaktioner.

## Datasettet

Vi använder ett riktigt dataset med ~284 000 kreditkortstransaktioner.

Varje rad är en transaktion med:
- Olika anonymiserade säkerhetsfunktioner (V1–V28)
- Beloppet (Amount)
- Klass: **0 = Legitim**, **1 = Bedräglig**

⚠️ **Viktigt:** Det finns mycket få bedrägerier – bara ~0,17% av alla transaktioner är bedrägliga!
Det innebär att en modell som alltid gissar 'legitimt' redan får 99,8% rätt – men är fullständigt värdelös.

## Steg 1: Ladda bedrägeridatasettet

In [ ]:
# Ladda data från internet
fraud_data = pd.read_csv(
    'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv'
)

print(f"Datamängd:  {len(fraud_data)} transaktioner")
print(f"Kolumner:   {fraud_data.shape[1]}")
print("\nFörsta 5 rader:")
print(fraud_data.head())

## Steg 2: Undersök klassfördelningen

Hur många bedrägerier finns det egentligen?

In [ ]:
# Räkna klassfördelning
class_counts = fraud_data['Class'].value_counts()

print("Klassfördelning:")
print(f"  Legitima:   {class_counts[0]} ({class_counts[0]/len(fraud_data)*100:.2f}%)")
print(f"  Bedrägliga: {class_counts[1]} ({class_counts[1]/len(fraud_data)*100:.2f}%)")

# Visualisera
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

class_counts.plot(kind='bar', ax=axes[0])
axes[0].set_title('Antal transaktioner per klass')
axes[0].set_ylabel('Antal')
axes[0].set_xticklabels(['Legitima', 'Bedrägliga'], rotation=0)

class_counts.plot(kind='pie', autopct='%1.2f%%', ax=axes[1],
                  labels=['Legitima', 'Bedrägliga'])
axes[1].set_title('Andel av totalt')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n⚠️ Märk att det är MYCKET färre bedrägerier än legitima transaktioner!")

## Steg 3: Förbered data

Vi delar upp data i träning och test precis som tidigare.

In [ ]:
# Ta bort tidskolumnen och separera features från target
X_fraud = fraud_data.drop(['Time', 'Class'], axis=1)  # Features
y_fraud = fraud_data['Class']                         # Target (0 eller 1)

# Dela data: 80% träning, 20% test
X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42
)

print(f"Träningsdata: {len(X_train_fraud)} transaktioner")
print(f"Testdata:     {len(X_test_fraud)} transaktioner")

## Steg 4: Träna modell för bedrägeridetektering

Den här modellen är binär – den svarar bara ja eller nej: är det bedrägeri?

- `objective: binary:logistic` – binär klassificering (ja/nej)
- `eval_metric: auc` – ett bättre mätvärde än noggrannhet när data är obalanserad

In [ ]:
# Konvertera till XGBoost-format
dtrain_fraud = xgb.DMatrix(X_train_fraud, label=y_train_fraud)
dtest_fraud  = xgb.DMatrix(X_test_fraud,  label=y_test_fraud)

# Sätt parametrar
param_fraud = {
    'max_depth': 4,
    'eta': 0.1,
    'objective': 'binary:logistic',  # Binär klassificering
    'eval_metric': 'auc'             # AUC är bättre än accuracy för obalanserad data
}

# Träna modellen
model_fraud = xgb.train(param_fraud, dtrain_fraud, num_boost_round=100)

print("Bedrägerimodellen är tränad!")

## Steg 5: Testa modellen

Vi gör förutsägelser och mäter noggrannheten – men kom ihåg varningen om klassoimbalans!

In [ ]:
# Förutsäg på testdata (modellen ger sannolikheter mellan 0 och 1)
fraud_predictions = model_fraud.predict(dtest_fraud)

# Klassificera: sannolikhet > 0.5 = bedrägeri
fraud_class = (fraud_predictions > 0.5).astype(int)

# Räkna rätt svar
correct_fraud  = (fraud_class == y_test_fraud.values).sum()
total_fraud    = len(y_test_fraud)
accuracy_fraud = correct_fraud / total_fraud

print(f"Noggrannhet: {accuracy_fraud*100:.2f}%")
print()
print("⚠️ Men tänk på: En modell som alltid gissar 'legitimt' får redan 99.8% rätt!")
print("   Därför måste vi titta på andra mätvärden – se nästa steg.")

## Steg 6: Förvirringmatris för bedrägeridetektering

Vi tittar på de olika typerna av rätt och fel:

| | Förutsagd: Legitim | Förutsagd: Bedrägeri |
|---|---|---|
| **Faktisk: Legitim** | ✅ Sann negativ (TN) | ❌ Falsk positiv (FP) – falskt larm |
| **Faktisk: Bedrägeri** | ❌ Falsk negativ (FN) – missat bedrägeri | ✅ Sann positiv (TP) |

In [ ]:
# Skapa förvirringmatris
cm_fraud = confusion_matrix(y_test_fraud, fraud_class)

# Rita den
plt.figure(figsize=(8, 6))
sns.heatmap(cm_fraud, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Legitim', 'Bedrägeri'],
            yticklabels=['Legitim', 'Bedrägeri'])
plt.title('Förvirringmatris – Bedrägeridetektering')
plt.ylabel('Faktisk klass')
plt.xlabel('Förutsagd klass')
plt.tight_layout()
plt.show()

# Beräkna viktiga mätvärden
TN = cm_fraud[0, 0]  # Legitima vi klassificerade rätt
FP = cm_fraud[0, 1]  # Falskt larm (legitim klassad som bedrägeri)
FN = cm_fraud[1, 0]  # Missade bedrägerier
TP = cm_fraud[1, 1]  # Bedrägerier vi hittade

sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0  # Andel hittade bedrägerier
precision   = TP / (TP + FP) if (TP + FP) > 0 else 0  # Andel korrekta larm

print(f"Bedrägerier hittade (känslighet): {TP} av {TP + FN} ({sensitivity*100:.1f}%)")
print(f"Falska larm (falskt positiva):    {FP} av {TN + FP} ({FP/(TN+FP)*100:.2f}%)")
print(f"Precision:                        {precision*100:.1f}%")

---

# Frågor att tänka på

1. **Irisklassificering:**
   - Var modellen helt korrekt? Om inte – vilka arter förväxlade den?
   - Varför tror du det?

2. **Bedrägeridetektering:**
   - Vilken typ av fel är värre: falskt larm eller missade bedrägerier?
   - Hur kan vi förbättra modellen för att hitta fler bedrägerier?

3. **Hyperparametrar:**
   - Prova att ändra `max_depth` eller `eta` i modellerna ovan. Blir de bättre eller sämre?
   - Vad tror du dessa parametrar gör med modellen?

4. **Klassoimbalans:**
   - Varför är noggrannhet (accuracy) ett dåligt mätvärde när data är obalanserad?
   - Vilket mätvärde tycker du är viktigast för en bedrägeridetekteringsmodell?